In [ ]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
from tqdm import tqdm
import re
import ast

In [ ]:
url = 'https://www.timeanddate.com/scripts/cityajax.php?n=usa/new-york&mode=historic&hd=20260202&month=2&year=2026&json=1'

page = requests.get(url)
soup = BeautifulSoup(page.text, 'html.parser')
text = soup.get_text()
text = text.replace('&lt;', '<').replace('&gt;', '>').replace('&quot;', '"')

json_like = re.search(r'\[\{.*\}\]', text)
if json_like:
    cleaned_json = json_like.group(0)
    cleaned_json = cleaned_json.replace('\"', '"').replace('\\/', '/')
    print(cleaned_json[:500])

[{c:[{h:"00:51Mon, 2 Feb</span>"},{s:"wt-ic",h:""},{h:"-9 °C"},{s:"small",h:"Clear."},{s:"sep",h:"13 km/h"},{h:"↑</span>"},{h:"56%"},{s:"sep",h:"1013 mbar"},{h:"16 km"}]},{c:[{h:"01:51"},{s:"wt-ic",h:""},{h:"-9 °C"},{s:"small",h:"Clear."},{s:"sep",h:"13 km/h"},{h:"↑</span>"},{h:"51%"},{s:"sep",h:"1014 mbar"},{h:"16 km"}]},{c:[{h:"02:51"},{s:"wt-ic",h:""},{h:"-9 °C"},{s:"small",h:"Clear."},{s:"sep",h:"11 km/h"},{h:"↑</span>"},{h:"46%"},{s:"sep",h:"1014 mbar"},{h:"16 km"}]},{c:[{h:"03:51"},{s:"wt-


In [ ]:

text = text.replace('&lt;', '<').replace('&gt;', '>').replace('&quot;', '"').replace('\\', '')
blocks = re.findall(r'\{c:\[(.*?)\]\}', text, re.DOTALL)

rows = []
for block in blocks:
    cells = re.findall(r'h:"(.*?)"', block, re.DOTALL)
    cells = [re.sub(r'<.*?>', '', c).strip() for c in cells]
    while len(cells) < 9:
        cells.append(None)
    rows.append(cells)

columns = ['Time', 'Icon', 'Temperature', 'Condition', 'Wind', 'Wind Direction', 'Humidity', 'Pressure', 'Visibility']
df = pd.DataFrame(rows, columns=columns)
df


,Time,Icon,Temperature,Condition,Wind,Wind Direction,Humidity,Pressure,Visibility
0,"00:51Mon, 2 Feb",,-9 °C,Clear.,13 km/h,↑,56%,1013 mbar,16 km
1,01:51,,-9 °C,Clear.,13 km/h,↑,51%,1014 mbar,16 km
2,02:51,,-9 °C,Clear.,11 km/h,↑,46%,1014 mbar,16 km
3,03:51,,-9 °C,Clear.,7 km/h,↑,45%,1014 mbar,16 km
4,04:51,,-9 °C,Clear.,17 km/h,↑,47%,1014 mbar,16 km
5,05:51,,-9 °C,Clear.,17 km/h,↑,49%,1015 mbar,16 km
6,06:51,,-8 °C,Clear.,9 km/h,↑,47%,1015 mbar,16 km
7,07:51,,-8 °C,Sunny.,15 km/h,↑,49%,1016 mbar,16 km
8,08:51,,-7 °C,Sunny.,17 km/h,↑,45%,1016 mbar,16 km
9,09:51,,-5 °C,Sunny.,N/A,↑,42%,1016 mbar,16 km


In [25]:
df_final= pd.DataFrame()
for month in tqdm(range(1, 13)):
    for day in tqdm(range(1, 32)):
        try:
            url = f'https://www.timeanddate.com/scripts/cityajax.php?n=usa/new-york&mode=historic&hd=2024{month:02d}{day:02d}&month={month:02d}&year=2024&json=1'

            page = requests.get(url)
            page.raise_for_status()

            soup = BeautifulSoup(page.text, 'html.parser')
            text = soup.get_text()
            text = text.replace('&lt;', '<').replace('&gt;', '>').replace('&quot;', '"')
            if "No data available for the given date" in soup.get_text():
                print(f"⚠️ No data available for {month}-{day}. Skipping.")
                continue

            json_like = re.search(r'\[\{.*\}\]', text)
            if json_like:
                cleaned_json = json_like.group(0)
                cleaned_json = cleaned_json.replace('\"', '"').replace('\\/', '/')

            text = text.replace('&lt;', '<').replace('&gt;', '>').replace('&quot;', '"').replace('\\', '')
            blocks = re.findall(r'\{c:\[(.*?)\]\}', text, re.DOTALL)

            rows = []
            for block in blocks:
                cells = re.findall(r'h:"(.*?)"', block, re.DOTALL)
                cells = [re.sub(r'<.*?>', '', c).strip() for c in cells]
                while len(cells) < 9:
                    cells.append(None)
                rows.append(cells)

            columns = ['Time', 'Icon', 'Temperature', 'Condition', 'Wind', 'Wind Direction', 'Humidity', 'Pressure', 'Visibility']
            df = pd.DataFrame(rows, columns=columns)
            df["day"] = day
            df["month"] = month
            df_final = pd.concat([df_final, df], ignore_index=True)
        except requests.exceptions.RequestException as e:
                  print(f"❌ Request failed for {month}-{day}: {e}")
                  continue
        except Exception as e:
                  print(f"❌ Unexpected error on {month}-{day}: {e}")
                  continue

 97%|█████████▋| 30/31 [00:07<00:00,  4.65it/s]

⚠️ No data available for 2-30. Skipping.



 17%|█▋        | 2/12 [00:14<01:15,  7.58s/it]

⚠️ No data available for 2-31. Skipping.



 33%|███▎      | 4/12 [00:29<00:59,  7.43s/it]

...


 75%|███████▌  | 9/12 [01:07<00:22,  7.45s/it]

⚠️ No data available for 9-31. Skipping.



 92%|█████████▏| 11/12 [01:20<00:07,  7.01s/it]

⚠️ No data available for 11-31. Skipping.



100%|██████████| 12/12 [01:27<00:00,  7.30s/it]


In [26]:
df_final

,Time,Icon,Temperature,Condition,Wind,Wind Direction,Humidity,Pressure,Visibility,day,month
0,"00:51Mon, 1 Jan",,6 °C,Mostly cloudy.,11 km/h,↑,55%,1016 mbar,16 km,1,1
1,01:51,,6 °C,Overcast.,7 km/h,↑,57%,1016 mbar,16 km,1,1
2,02:51,,5 °C,Overcast.,No wind,↑,60%,1016 mbar,16 km,1,1
3,03:51,,5 °C,Overcast.,6 km/h,↑,62%,1015 mbar,16 km,1,1
4,04:51,,4 °C,Partly cloudy.,No wind,↑,68%,1015 mbar,16 km,1,1
...,...,...,...,...,...,...,...,...,...,...,...
9649,20:51,,7 °C,Rain. Fog.,6 km/h,↑,86%,1003 mbar,6 km,31,12
9650,21:56,,7 °C,Heavy rain. Fog.,7 km/h,↑,89%,1003 mbar,2 km,31,12
9651,22:33,,7 °C,Rain. Fog.,9 km/h,↑,90%,1002 mbar,3 km,31,12
9652,22:51,,7 °C,Rain. Fog.,19 km/h,↑,93%,1000 mbar,8 km,31,12


In [27]:
df_final["year"] = 2024
df_final["Time_clean"] = df_final["Time"].str.extract(r"(\d{2}:\d{2})")
df_final["Timestamp"] = pd.to_datetime(
    df_final["year"].astype(str) + "-" +
    df_final["month"].astype(str) + "-" +
    df_final["day"].astype(str) + " " +
    df_final["Time_clean"],
    format="%Y-%m-%d %H:%M"
)
df_final.drop(columns=["Time_clean"], inplace=True)

In [28]:
df_final

,Time,Icon,Temperature,Condition,Wind,Wind Direction,Humidity,Pressure,Visibility,day,month,year,Timestamp
0,"00:51Mon, 1 Jan",,6 °C,Mostly cloudy.,11 km/h,↑,55%,1016 mbar,16 km,1,1,2024,2024-01-01 00:51:00
1,01:51,,6 °C,Overcast.,7 km/h,↑,57%,1016 mbar,16 km,1,1,2024,2024-01-01 01:51:00
2,02:51,,5 °C,Overcast.,No wind,↑,60%,1016 mbar,16 km,1,1,2024,2024-01-01 02:51:00
3,03:51,,5 °C,Overcast.,6 km/h,↑,62%,1015 mbar,16 km,1,1,2024,2024-01-01 03:51:00
4,04:51,,4 °C,Partly cloudy.,No wind,↑,68%,1015 mbar,16 km,1,1,2024,2024-01-01 04:51:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9649,20:51,,7 °C,Rain. Fog.,6 km/h,↑,86%,1003 mbar,6 km,31,12,2024,2024-12-31 20:51:00
9650,21:56,,7 °C,Heavy rain. Fog.,7 km/h,↑,89%,1003 mbar,2 km,31,12,2024,2024-12-31 21:56:00
9651,22:33,,7 °C,Rain. Fog.,9 km/h,↑,90%,1002 mbar,3 km,31,12,2024,2024-12-31 22:33:00
9652,22:51,,7 °C,Rain. Fog.,19 km/h,↑,93%,1000 mbar,8 km,31,12,2024,2024-12-31 22:51:00


In [29]:
df_final["Timestamp"] = (
    pd.to_datetime(df_final["Timestamp"], format="%Y-%m-%d %H:%M")
      .dt.round("30min")
      .dt.strftime("%Y-%m-%d %H:%M")
)
df_final

,Time,Icon,Temperature,Condition,Wind,Wind Direction,Humidity,Pressure,Visibility,day,month,year,Timestamp
0,"00:51Mon, 1 Jan",,6 °C,Mostly cloudy.,11 km/h,↑,55%,1016 mbar,16 km,1,1,2024,2024-01-01 01:00
1,01:51,,6 °C,Overcast.,7 km/h,↑,57%,1016 mbar,16 km,1,1,2024,2024-01-01 02:00
2,02:51,,5 °C,Overcast.,No wind,↑,60%,1016 mbar,16 km,1,1,2024,2024-01-01 03:00
3,03:51,,5 °C,Overcast.,6 km/h,↑,62%,1015 mbar,16 km,1,1,2024,2024-01-01 04:00
4,04:51,,4 °C,Partly cloudy.,No wind,↑,68%,1015 mbar,16 km,1,1,2024,2024-01-01 05:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9649,20:51,,7 °C,Rain. Fog.,6 km/h,↑,86%,1003 mbar,6 km,31,12,2024,2024-12-31 21:00
9650,21:56,,7 °C,Heavy rain. Fog.,7 km/h,↑,89%,1003 mbar,2 km,31,12,2024,2024-12-31 22:00
9651,22:33,,7 °C,Rain. Fog.,9 km/h,↑,90%,1002 mbar,3 km,31,12,2024,2024-12-31 22:30
9652,22:51,,7 °C,Rain. Fog.,19 km/h,↑,93%,1000 mbar,8 km,31,12,2024,2024-12-31 23:00


In [30]:
df_final = df_final.drop(['Icon', 'Wind Direction', 'Time', 'day', 'month', 'year'], axis=1)

In [31]:
df_final

,Temperature,Condition,Wind,Humidity,Pressure,Visibility,Timestamp
0,6 °C,Mostly cloudy.,11 km/h,55%,1016 mbar,16 km,2024-01-01 01:00
1,6 °C,Overcast.,7 km/h,57%,1016 mbar,16 km,2024-01-01 02:00
2,5 °C,Overcast.,No wind,60%,1016 mbar,16 km,2024-01-01 03:00
3,5 °C,Overcast.,6 km/h,62%,1015 mbar,16 km,2024-01-01 04:00
4,4 °C,Partly cloudy.,No wind,68%,1015 mbar,16 km,2024-01-01 05:00
...,...,...,...,...,...,...,...
9649,7 °C,Rain. Fog.,6 km/h,86%,1003 mbar,6 km,2024-12-31 21:00
9650,7 °C,Heavy rain. Fog.,7 km/h,89%,1003 mbar,2 km,2024-12-31 22:00
9651,7 °C,Rain. Fog.,9 km/h,90%,1002 mbar,3 km,2024-12-31 22:30
9652,7 °C,Rain. Fog.,19 km/h,93%,1000 mbar,8 km,2024-12-31 23:00


In [32]:
df_final.to_csv('weather_data2024.csv', index=False)